Now we have the papers of interest filtered (~ 17,500), next step is to extract theory papers.

Note: using zero-shot NLP models from Huggingface, switch to conda environment with torch-gpu installed. 

**Update**: probably not actually needed.

We can directly skip to step 3, to extract opinions.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/SC_related_cites_010.csv")

df = df.dropna(); print(len(df))
df[1000:1010]

17503


,abstract,concepts_scores,doi,id,journal,times_cited,title,year
1121,The free energy is calculated for the various ...,"[{'concept': 'magnetic order', 'relevance': 0....",10.1103/physrevlett.46.49,pub.1060786191,"{'id': 'jour.1018277', 'title': 'Physical Revi...",165,Possible Coexisting Superconducting and Magnet...,1981
1122,Neutron scattering has been used to investigat...,"[{'concept': 'superconducting phase', 'relevan...",10.1103/physrevlett.46.368,pub.1060786151,"{'id': 'jour.1018277', 'title': 'Physical Revi...",112,Competition between Ferromagnetism and Superco...,1981
1123,Pressure-induced high-temperature superconduct...,"[{'concept': 'high-Tc superconductors', 'relev...",10.1103/physrevlett.46.280,pub.1060786124,"{'id': 'jour.1018277', 'title': 'Physical Revi...",81,Observation of the Transition from Semiconduct...,1981
1124,"The two anomalous rare-earth ternaries, Eu1.2M...","[{'concept': 'high-pressure studies', 'relevan...",10.1103/physrevlett.46.276,pub.1060786123,"{'id': 'jour.1018277', 'title': 'Physical Revi...",83,High-Pressure Study of the Anomalous Rare-Eart...,1981
1125,From the critical-field data on the strongly P...,"[{'concept': 'A15 superconductors', 'relevance...",10.1103/physrevlett.46.1598,pub.1060786050,"{'id': 'jour.1018277', 'title': 'Physical Revi...",88,Pauli Limiting and the Possibility of Spin Flu...,1981
1126,The Kosterlitz-Thouless theory applied to two-...,"[{'concept': 'vortex unbinding', 'relevance': ...",10.1103/physrevlett.46.148,pub.1060786008,"{'id': 'jour.1018277', 'title': 'Physical Revi...",65,Search for Vortex Unbinding in Two-Dimensional...,1981
1127,Superconductivity is found to exist in the ter...,"[{'concept': 'ternary intermetallics', 'releva...",10.1103/physrevb.24.6715,pub.1060529912,"{'id': 'jour.1320488', 'title': 'Physical Revi...",83,Superconductivity in the ternary intermetallic...,1981
1128,The destruction of superconductivity in granul...,[{'concept': 'destruction of superconductivity...,10.1103/physrevb.24.6353,pub.1060529868,"{'id': 'jour.1320488', 'title': 'Physical Revi...",125,Destruction of superconductivity in granular a...,1981
1129,The question of coherent interfaces between mu...,[{'concept': 'superconducting transition tempe...,10.1103/physrevb.24.6193,pub.1060529849,"{'id': 'jour.1320488', 'title': 'Physical Revi...",60,X-ray scattering from multilayers of NbCu,1981
1130,For granular metallic films or two-dimensional...,[{'concept': 'two-dimensional arrays of Joseph...,10.1103/physrevb.24.5063,pub.1060529694,"{'id': 'jour.1320488', 'title': 'Physical Revi...",202,Quantum fluctuations in two-dimensional superc...,1981


In [3]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Device set to use cuda:0


Zero-shot classifier set-up done!

In [ ]:
from tqdm.notebook import tqdm
from datasets import Dataset
dataset = Dataset.from_pandas(df[['abstract']])

tqdm.pandas()

def classify_batch(batch, labels=['Theoretical', 'Experimental']):
    results = classifier(batch['abstract'], candidate_labels=labels)
    return {
        'is_theoretical': [res['labels'][0] == 'Theoretical' for res in results],
        'confidence': [res['scores'][0] for res in results]
    }

# Map over dataset
dataset = dataset.map(classify_batch, batched=True, batch_size=8)  # adjust batch size depending on VRAM

# Convert back to DataFrame
df = df.join(dataset.to_pandas()[['is_theoretical', 'confidence']])

Map:   0%|          | 0/17503 [00:00<?, ? examples/s]

/home/vipandyc/miniconda3/envs/LLM/lib/python3.9/site-packages/torch/utils/data/dataloader.py:641: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0x7f1f24780130> was reported to be 8(when accessing len(dataloader)), but 9 samples have been fetched. 
  warnings.warn(warn_msg)
/home/vipandyc/miniconda3/envs/LLM/lib/python3.9/site-packages/torch/utils/data/dataloader.py:641: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0x7f1f24780130> was reported to be 8(when accessing len(dataloader)), but 10 samples have been fetched. 
  warnings.warn(warn_msg)
/home/vipandyc/miniconda3/envs/LLM/lib/python3.9/site-packages/torch/utils/data/dataloader.py:641: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0x7f1f24780130> was reported to be 8(when accessing len(dataloader)), but 11 samples have been fetched. 
  warnings.warn(warn_msg)

In [11]:
df = df[df['is_theoretical'] == True]
df.to_csv("../data/SC_theory_highlights.csv", index=False)
